# Lezione 2 — Visione artificiale: reti neurali e convoluzioni

Corso **Fondamenti di TensorFlow da zero** · DIH DATAlife

## Obiettivi della lezione

- Addestrare un classificatore di immagini con una rete densa (Fashion MNIST)
- Spiegare che cos'è una convoluzione e un filtro, e perché migliorano la visione artificiale
- Costruire una rete neurale convoluzionale (CNN) semplice e confrontare la sua precisione con la rete densa
- Interpretare l'effetto dei livelli di pooling e di diversi livelli convoluzionali

Questo notebook combina teoria (celle Markdown) e codice funzionale in TensorFlow 2.x, pronto per essere eseguito in Google Colab senza installare nulla di aggiuntivo. Consulta i capitoli 3 e 4 del manuale del corso per la spiegazione completa.

## Parte 1 · Una rete densa per "vedere" immagini

Un computer non vede abbigliamento: vede una griglia di numeri. Un'immagine in scala di grigi di 28x28 pixel è una matrice di 28x28 valori tra 0 e 255, uno per pixel. La visione artificiale con Machine Learning (ML) consiste nell'insegnare al modello migliaia di esempi etichettati affinché impari da solo quali combinazioni di quei numeri corrispondono a ogni classe, invece di programmare regole a mano.

Andiamo ad addestrare una prima rete neurale densa con il dataset **Fashion MNIST**: 70.000 immagini di indumenti (60.000 per l'addestramento e 10.000 per i test), in scala di grigi, di 28x28 pixel, distribuiti in dieci categorie.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

print('Versión de TensorFlow:', tf.__version__)

In [ ]:
# Cargamos Fashion MNIST directamente desde la API de datasets de Keras
fashion_mnist = tf.keras.datasets.fashion_mnist
(training_images, training_labels), (test_images, test_labels) = fashion_mnist.load_data()

class_names = ['Camiseta', 'Pantalón', 'Jersey', 'Vestido', 'Abrigo',
               'Sandalia', 'Camisa', 'Zapatilla', 'Bolso', 'Botín']

print('Forma de las imágenes de entrenamiento:', training_images.shape)
print('Número de etiquetas de entrenamiento:', len(training_labels))

plt.figure()
plt.imshow(training_images[0], cmap='gray')
plt.title(f'Etiqueta: {training_labels[0]} ({class_names[training_labels[0]]})')
plt.colorbar()
plt.grid(False)
plt.show()

### Normalizzazione dei dati

Ogni pixel vale tra 0 e 255. Le reti si addestrano meglio quando i valori in ingresso sono in un intervallo piccolo e omogeneo, quindi **normalizziamo** dividendo per 255. È importante applicare esattamente la stessa trasformazione sia al set di addestramento che a quello di test.

In [ ]:
training_images_norm = training_images / 255.0
test_images_norm = test_images / 255.0

### Strati, attivazioni e softmax

La rete densa ha tre componenti:

- **Flatten**: converte l'immagine 28x28 in un vettore di 784 valori, senza apprendere nulla
- **Dense(128, activation='relu')**: uno strato nascosto di 128 neuroni. L'attivazione **ReLU** (Rectified Linear Unit, Unità Lineare Rettificata) lascia passare i valori positivi tali e quali e trasforma in zero i negativi
- **Dense(10, activation='softmax')**: lo strato di uscita, con un neurone per classe. **Softmax** convertisce i valori di uscita in probabilità che sommano 1, quindi la classe con la probabilità più alta è la previsione del modello

Compiliamo il modello con l'ottimizzatore `adam`, la funzione di perdita `sparse_categorical_crossentropy` (perché le etichette sono numeri interi) e la metrica `accuracy`.

In [ ]:
modelo_denso = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

modelo_denso.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])

modelo_denso.fit(training_images_norm, training_labels, epochs=5)

In [ ]:
loss_denso, accuracy_denso = modelo_denso.evaluate(test_images_norm, test_labels)
print(f'\nExactitud del modelo denso en el conjunto de prueba: {accuracy_denso:.4f}')

predicciones_denso = modelo_denso.predict(test_images_norm)
print('Probabilidades para la primera imagen de prueba:', predicciones_denso[0])
print('Clase predicha:', np.argmax(predicciones_denso[0]), '- Clase real:', test_labels[0])

### Normalizzazione dei dati

Ogni pixel ha un valore tra 0 e 255. Le reti si addestrano meglio quando i valori di input sono in un intervallo piccolo e omogeneo, quindi **normalizziamo** dividendo per 255. È importante applicare esattamente la stessa trasformazione sia al set di addestramento che a quello di test.

## Parte 2 · Che cos'è una convoluzione

Una **convoluzione** fa scorrere un filtro (una piccola matrice di numeri, chiamata anche kernel) su tutta l'immagine. In ogni posizione, il filtro combina il pixel centrale con i suoi vicini: moltiplica ogni pixel per il valore del filtro che gli corrisponde e somma i risultati. Quello è il nuovo valore del pixel nell'immagine trasformata.

Verifichiamolo con un'immagine di esempio (la classica "ascent" fornita da `scipy`) e un filtro che risalta i bordi orizzontali.

In [ ]:
try:
    from scipy import datasets as scipy_datasets
    imagen_original = scipy_datasets.ascent()
except ImportError:
    from scipy import misc
    imagen_original = misc.ascent()

plt.gray()
plt.grid(False)
plt.axis('off')
plt.imshow(imagen_original)
plt.title('Imagen original')
plt.show()

print('Forma de la imagen:', imagen_original.shape)

In [ ]:
def aplicar_filtro(imagen, filtro, peso=1.0):
    """Aplica manualmente un filtro de convolución 3x3 a una imagen en escala de grises."""
    imagen_filtrada = np.copy(imagen)
    size_x, size_y = imagen.shape

    for x in range(1, size_x - 1):
        for y in range(1, size_y - 1):
            convolucion = 0.0
            for fx in range(3):
                for fy in range(3):
                    convolucion += imagen[x + fx - 1, y + fy - 1] * filtro[fx][fy]
            convolucion *= peso
            convolucion = max(0, min(255, convolucion))
            imagen_filtrada[x, y] = convolucion

    return imagen_filtrada

# Filtro que resalta bordes horizontales
filtro_horizontal = [[-1, -2, -1],
                      [ 0,  0,  0],
                      [ 1,  2,  1]]

imagen_convolucionada = aplicar_filtro(imagen_original, filtro_horizontal)

plt.gray()
plt.grid(False)
plt.axis('off')
plt.imshow(imagen_convolucionada)
plt.title('Después de aplicar el filtro (bordes horizontales)')
plt.show()

### Strati, attivazioni e softmax

La rete densa ha tre componenti:

- **Flatten**: converte l'immagine 28x28 in un vettore di 784 valori, senza imparare nulla
- **Dense(128, activation='relu')**: uno strato nascosto di 128 neuroni. L'attivazione **ReLU** (Rectified Linear Unit, Unità Lineare Rettificata) lascia passare i valori positivi così come sono e converte in zero i negativi
- **Dense(10, activation='softmax')**: lo strato di output, con un neurone per classe. **Softmax** converte i valori di output in probabilità che sommano a 1, così la classe con la probabilità più alta è la predizione del modello

Compiliamo il modello con l'ottimizzatore `adam`, la funzione di perdita `sparse_categorical_crossentropy` (perché le etichette sono numeri interi) e la metrica `accuracy`.

In [ ]:
def max_pooling(imagen, tamano=2):
    """Aplica max pooling de tamano x tamano sobre una imagen en escala de grises."""
    size_x, size_y = imagen.shape
    nueva_x, nueva_y = size_x // tamano, size_y // tamano
    imagen_reducida = np.zeros((nueva_x, nueva_y))

    for x in range(0, size_x - 1, tamano):
        for y in range(0, size_y - 1, tamano):
            bloque = imagen[x:x + tamano, y:y + tamano]
            imagen_reducida[x // tamano, y // tamano] = np.max(bloque)

    return imagen_reducida

imagen_pooled = max_pooling(imagen_convolucionada, tamano=2)

print('Tamaño antes del pooling:', imagen_convolucionada.shape)
print('Tamaño después del pooling:', imagen_pooled.shape)

plt.gray()
plt.grid(False)
plt.axis('off')
plt.imshow(imagen_pooled)
plt.title('Después de max pooling 2x2')
plt.show()

L'immagine risultante occupa un quarto dei pixel dell'originale, ma i tratti che il filtro aveva risaltato restano riconoscibili. Questo è il doppio vantaggio del pooling: meno calcolo per gli strati successivi e più robustezza rispetto a piccoli spostamenti dell'oggetto nell'immagine.

## Parte 3 · Costruire una CNN completa in Keras

In Keras non applichiamo le convoluzioni e il pooling a mano: usiamo gli strati `Conv2D` e `MaxPooling2D`. Una CNN tipica concatena vari blocchi di convoluzione + pooling per estrarre caratteristiche, e termina con `Flatten` e strati `Dense` per classificare, proprio come la rete densa della Parte 1.

`Conv2D` si aspetta immagini con un canale esplicito, quindi dobbiamo aggiungere quella dimensione a Fashion MNIST: da `(28, 28)` a `(28, 28, 1)`.

In [ ]:
training_images_cnn = training_images_norm.reshape(-1, 28, 28, 1)
test_images_cnn = test_images_norm.reshape(-1, 28, 28, 1)

modelo_cnn = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

modelo_cnn.summary()

In [ ]:
modelo_cnn.compile(optimizer='adam',
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])

modelo_cnn.fit(training_images_cnn, training_labels, epochs=5)

In [ ]:
loss_cnn, accuracy_cnn = modelo_cnn.evaluate(test_images_cnn, test_labels)

print(f'Exactitud red densa (Parte 1): {accuracy_denso:.4f}')
print(f'Exactitud CNN       (Parte 3): {accuracy_cnn:.4f}')

### I limiti della rete densa

Questa rete densa impara direttamente dai pixel grezzi. Funziona bene con Fashion MNIST perché tutte le immagini sono centrate, in scala di grigi e hanno esattamente le stesse dimensioni, ma questa stessa dipendenza è il suo limite: non ha alcun concetto di "bordo" o "forma", memorizza solo modelli di pixel. Con una foto reale, con uno sfondo o con l'indumento spostato, questa rete fallisce.

Per superare questa limitazione abbiamo bisogno che il modello impari prima a rilevare **tratti visivi** (bordi, curve, texture) prima di classificare. A questo serve la seconda parte del notebook: le convoluzioni.

## Parte 2 · Che cos'è una convoluzione

Una **convoluzione** fa scorrere un filtro (una piccola matrice di numeri, chiamata anche kernel) su tutta l'immagine. Ad ogni posizione, il filtro combina il pixel centrale con i suoi vicini: moltiplica ogni pixel per il valore del filtro che gli corrisponde e somma i risultati. Quel è il nuovo valore del pixel nell'immagine trasformata.

Verifichiamolo con un'immagine di esempio (il classico "ascent" che viene con `scipy`) e un filtro che evidenzia i bordi orizzontali.

In [ ]:
def plot_imagen(indice, predicciones, etiquetas_reales, imagenes):
    prediccion = predicciones[indice]
    etiqueta_real = etiquetas_reales[indice]
    imagen = imagenes[indice].reshape(28, 28)

    clase_predicha = np.argmax(prediccion)
    color = 'blue' if clase_predicha == etiqueta_real else 'red'

    plt.grid(False)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(imagen, cmap=plt.cm.binary)
    plt.xlabel('{} {:2.0f}% (real: {})'.format(
        class_names[clase_predicha],
        100 * np.max(prediccion),
        class_names[etiqueta_real]),
        color=color)


def plot_probabilidades(indice, predicciones, etiquetas_reales):
    prediccion = predicciones[indice]
    etiqueta_real = etiquetas_reales[indice]

    plt.grid(False)
    plt.xticks(range(10))
    plt.yticks([])
    barras = plt.bar(range(10), prediccion, color='#777777')
    plt.ylim([0, 1])

    clase_predicha = np.argmax(prediccion)
    barras[clase_predicha].set_color('red')
    barras[etiqueta_real].set_color('blue')


predicciones_cnn = modelo_cnn.predict(test_images_cnn)

plt.figure(figsize=(6, 3))
plt.subplot(1, 2, 1)
plot_imagen(0, predicciones_cnn, test_labels, test_images_cnn)
plt.subplot(1, 2, 2)
plot_probabilidades(0, predicciones_cnn, test_labels)
plt.show()

In [ ]:
# Predicción sobre una única imagen "nueva": recuerda que Keras siempre espera un lote (batch)
imagen_nueva = test_images_cnn[1]
lote = np.expand_dims(imagen_nueva, axis=0)   # forma (1, 28, 28, 1)

prediccion_unica = modelo_cnn.predict(lote)
clase_predicha = np.argmax(prediccion_unica[0])

print(f'Clase predicha: {class_names[clase_predicha]} ({clase_predicha})')
print(f'Clase real: {class_names[test_labels[1]]} ({test_labels[1]})')
print(f'Confianza: {np.max(prediccion_unica[0]):.2%}')

## Riepilogo e prossimi passi

- Una rete densa classifica immagini apprendendo direttamente dai pixel grezzi, ma funziona bene solo con immagini centrate e della stessa dimensione
- Una convoluzione filtra l'immagine per risaltare tratti (bordi, texture); i valori del filtro vengono appresi durante l'addestramento, non vengono progettati a mano
- Il pooling riduce la dimensione dell'immagine conservando i tratti più importanti, alleggerendo il calcolo e dando robustezza rispetto agli spostamenti
- Una CNN combina blocchi `Conv2D` + `MaxPooling2D` per estrarre caratteristiche e termina con `Flatten` + `Dense` per classificare, proprio come la rete densa
- Interpretare l'uscita di un classificatore significa leggere le probabilità di softmax con `np.argmax`, non solo guardare se l'`accuracy` è salita

Ora apri il notebook di pratica di questa lezione (`clase02_practica_vision_por_computador_cnn.ipynb`) per modificare tu stesso l'architettura della CNN e misurare l'effetto sulla precisione.